In [1]:
from common_utils import *
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
import seaborn as sns


In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# os.chdir('/content/drive/MyDrive/SC4001_Group_Assignmnet_2')
set_seed()

In [3]:
# CrowdFlower
# dataset = load_dataset("csv", data_files="./dataset/text_emotion.csv")
# dataset = crowd_dataset.rename_column('content', 'text')
# dataset_dict = create_train_validation_test(crowd_dataset['train'])

# Wassa
dataset = load_dataset("csv", data_files="./dataset/wassa_combined_data.csv")
dataset = dataset.rename_column('tweet', 'text')
dataset_dict = create_train_validation_test(dataset['train'])

embedding_matrix = np.load(EMBEDDING_PATH)

with open(WORD2IDX_PATH, "r", encoding="utf-8") as f:
    word2idx = json.load(f)

Train size: 5592
Validation size: 799
Test size: 711


In [4]:
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(dataset_dict['train']['sentiment'])
val_labels = label_encoder.transform(dataset_dict['validation']['sentiment'])
test_labels = label_encoder.transform(dataset_dict['test']['sentiment'])
num_classes = len(label_encoder.classes_)
print(f"Number of sentiment classes: {num_classes}")
print(f"Emotion classes: {label_encoder.classes_}")

crowd_labels_dict = {
    'train': train_labels,
    'validation': val_labels,
    'test': test_labels
}

Number of sentiment classes: 4
Emotion classes: ['anger' 'fear' 'joy' 'sadness']


In [5]:
dataloaders_dict = create_dataloaders(dataset_dict, crowd_labels_dict, word2idx, batch_size=16, max_seq_length=100)

Created DataLoaders with 350 training batches, 50 validation batches, and 45 test batches.


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BiGRU_Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, embedding_matrix, dropout=0.5):
        super(BiGRU_Model, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = True 

        self.bigru1 = nn.GRU(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.dropout_bigru1 = nn.Dropout(0.5)
        self.bigru2 = nn.GRU(2 * hidden_dim, 64, bidirectional=True, batch_first=True)
        self.dropout_bigru2 = nn.Dropout(0.3)
        self.fc = nn.Linear(2 * 64, num_classes)
        self.layer_norm = nn.LayerNorm(2 * 64)

    def forward(self, x):
        x = self.embedding(x) 
        x, _ = self.bigru1(x) 
        x = self.dropout_bigru1(x)
        x, _ = self.bigru2(x)
        x = self.dropout_bigru2(x)
        x = x[:, -1, :] 
        x = self.layer_norm(x)
        x = self.fc(x)

        return x



In [8]:
num_epochs = 100
learning_rate = 0.0005
patience = 5  

model = BiGRU_Model(vocab_size=len(word2idx),  
                         embedding_dim=embedding_matrix.shape[1],
                         hidden_dim=256, 
                         num_classes=4, 
                         embedding_matrix=embedding_matrix)  

model.to(device)

criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.005)

# Early stopping
best_val_loss = float('inf') 
epochs_without_improvement = 0 
best_epoch = 0 
best_model_path = 'best_model.pth'

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloaders_dict['train']:
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / len(dataloaders_dict['train'])
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

    # Validation
    model.eval() 
    with torch.no_grad():
        val_loss = 0.0
        correct = 0
        total = 0
        for batch in dataloaders_dict['validation']:
            input_ids, labels = batch
            input_ids, labels = input_ids.to(device), labels.to(device)

            outputs = model(input_ids)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        val_loss /= len(dataloaders_dict['validation'])
        val_acc = 100 * correct / total
        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        best_epoch = epoch + 1 
        
        torch.save(model.state_dict(), best_model_path)
        print(f"Model saved at epoch {epoch + 1} with validation loss: {best_val_loss:.4f}")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping triggered after {epoch + 1} epochs (no improvement in {patience} epochs).")
        break

# Load the best model
model.load_state_dict(torch.load(best_model_path))
print(f"Loaded best model from epoch {best_epoch}")


Epoch [1/100], Loss: 1.4110, Accuracy: 28.24%
Validation Loss: 1.3925, Validation Accuracy: 31.16%
Model saved at epoch 1 with validation loss: 1.3925
Epoch [2/100], Loss: 1.3884, Accuracy: 29.81%
Validation Loss: 1.3948, Validation Accuracy: 22.15%
Epoch [3/100], Loss: 1.3818, Accuracy: 29.99%
Validation Loss: 1.3886, Validation Accuracy: 31.16%
Model saved at epoch 3 with validation loss: 1.3886
Epoch [4/100], Loss: 1.3798, Accuracy: 31.15%
Validation Loss: 1.3874, Validation Accuracy: 22.15%
Model saved at epoch 4 with validation loss: 1.3874
Epoch [5/100], Loss: 1.3797, Accuracy: 31.08%
Validation Loss: 1.3793, Validation Accuracy: 31.16%
Model saved at epoch 5 with validation loss: 1.3793
Epoch [6/100], Loss: 1.3791, Accuracy: 31.03%
Validation Loss: 1.3822, Validation Accuracy: 31.16%
Epoch [7/100], Loss: 1.3773, Accuracy: 31.65%
Validation Loss: 1.3794, Validation Accuracy: 31.16%
Epoch [8/100], Loss: 1.3773, Accuracy: 31.62%
Validation Loss: 1.3788, Validation Accuracy: 31.16%


In [9]:
model.eval()
all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    test_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloaders_dict['test']:
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        outputs = model(input_ids)
        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

        all_probabilities.extend(torch.softmax(outputs, dim=1).cpu().numpy())

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Test loss and accuracy
test_loss /= len(dataloaders_dict['test'])
test_acc = 100 * correct / total
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%")

# F1 Score 
f1 = f1_score(all_labels, all_predictions, average='macro')
print(f"F1 Score (Macro-average): {f1:.4f}")

class_f1 = f1_score(all_labels, all_predictions, average=None)
for i, f1_class in enumerate(class_f1):
    print(f"F1 Score for Class {i}: {f1_class:.4f}")

Test Loss: 1.3700, Test Accuracy: 33.19%
F1 Score (Macro-average): 0.1246
F1 Score for Class 0: 0.0000
F1 Score for Class 1: 0.4984
F1 Score for Class 2: 0.0000
F1 Score for Class 3: 0.0000


In [10]:
all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    test_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloaders_dict['validation']:
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        # Forward pass
        outputs = model(input_ids)
        loss = criterion(outputs, labels)

        # Track test loss and accuracy
        test_loss += loss.item()

        # Predicted class labels
        _, predicted = torch.max(outputs, 1)

        # Collect all predictions and labels for confusion matrix
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

        #Collect predicted probabilities for ROC curve
        all_probabilities.extend(torch.softmax(outputs, dim=1).cpu().numpy())

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate test loss and accuracy
test_loss /= len(dataloaders_dict['validation'])
test_acc = 100 * correct / total
print(f"val Loss: {test_loss:.4f}, val Accuracy: {test_acc:.2f}%")

# Calculate F1 Score (Macro-average)
f1 = f1_score(all_labels, all_predictions, average='macro')
print(f"val F1 Score (Macro-average): {f1:.4f}")

# Calculate F1 Score for each class (Class-specific F1)
class_f1 = f1_score(all_labels, all_predictions, average=None)
for i, f1_class in enumerate(class_f1):
    print(f"F1 Score for Class {i}: {f1_class:.4f}")

val Loss: 1.3771, val Accuracy: 31.16%
val F1 Score (Macro-average): 0.1188
F1 Score for Class 0: 0.0000
F1 Score for Class 1: 0.4752
F1 Score for Class 2: 0.0000
F1 Score for Class 3: 0.0000
